# Climate Trace Emissions

In [ ]:
import pandas as pd, geopandas as gpd

# After downloading from climatetrace.org/data
df_ct = pd.read_csv("climatetrace_emissions.csv")

# Columns include: lat, lon, sector, co2e_100yr (tonnes CO₂e/yr)
# Already in (lat, lon, value) format — minimal processing needed

df_ct = df_ct[["lat", "lon", "co2e_100yr", "sector"]].dropna()

In [1]:
pwd

'/lisc/home/user/gerardursin/pain-world'

In [10]:
import os
import glob
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────
# Use absolute paths — no ambiguity about working directory
BASE_DIR   = Path("/lisc/home/user/gerardursin/pain-world")
DATA_DIR   = BASE_DIR / "climatetrace"
OUTPUT_CSV = BASE_DIR / "climate_trace_compiled.csv"

TARGET_GAS  = "co2e_100yr"
YEAR_START  = "2020-01-01"
YEAR_END    = "2021-12-31"
CRS_OUT     = "EPSG:4326"

# Quick sanity check before doing anything else
print(f"DATA_DIR exists : {DATA_DIR.exists()}")
print(f"Files found     : {len(list(DATA_DIR.glob('*.csv')))}")
print(f"Output will go  : {OUTPUT_CSV}")

DATA_DIR exists : True
Files found     : 48
Output will go  : /lisc/home/user/gerardursin/pain-world/climate_trace_compiled.csv


In [15]:
KG_SUBSECTORS = {
    "enteric-fermentation-cattle-pasture-raster",
    "manure-left-on-pasture-cattle-raster",
}

def normalize_gas(s: str) -> str:
    return str(s).lower().replace("-", "").replace("_", "").replace(" ", "")

GAS_VARIANTS = {
    normalize_gas(g) for g in [
        "co2e_100yr", "co2e100yr", "co2e_100", "co2e100",
        "co2equivalent100", "co2e_ar6_100"
    ]
}

def find_file_pairs(root: Path) -> list[dict]:
    """All CSVs are flat in one folder — no subdirectories."""
    csvs  = sorted(root.glob("*.csv"))
    gpkgs = sorted(root.glob("*.gpkg"))

    # Single shared geopackage (geometries.gpkg)
    shared_gpkg = root / "geometries.gpkg"
    gpkg_path = str(shared_gpkg) if shared_gpkg.exists() else None

    pairs = []
    for csv_file in csvs:
        pairs.append({
            "csv":  str(csv_file),
            "gpkg": gpkg_path,
            "dir":  str(root),
        })

    print(f"  {len(pairs)} CSVs found")
    print(f"  Geopackage: {gpkg_path or 'NOT FOUND'}")
    return pairs


def load_geopackage(gpkg_path: str) -> gpd.GeoDataFrame | None:
    try:
        gdf = gpd.read_file(gpkg_path)
    except Exception as e:
        print(f"  ✗ Could not read geopackage: {e}")
        return None

    if gdf.crs is None:
        gdf = gdf.set_crs(CRS_OUT)
    else:
        gdf = gdf.to_crs(CRS_OUT)

    ref_col = _find_ref_col(gdf)
    centroids = gdf.geometry.centroid
    gdf["_lat"] = centroids.y
    gdf["_lon"] = centroids.x

    result = gdf[["_lat", "_lon", "geometry"]].copy()
    result["geometry_ref"] = gdf[ref_col].astype(str) if ref_col else None

    print(f"  📦 Geopackage: {len(result):,} features, ref_col='{ref_col}'")
    # Print columns so we can see what's in it
    print(f"     Columns: {list(gdf.columns[:10])}")
    return result


def _find_ref_col(gdf: gpd.GeoDataFrame) -> str | None:
    candidates = [
        "geometry_ref", "geom_ref", "source_id", "id",
        "asset_id", "fid", "GID_0", "GID_1", "GID_2"
    ]
    cols_lower = {c.lower(): c for c in gdf.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None


def load_csv(filepath: str) -> pd.DataFrame | None:
    try:
        df = pd.read_csv(filepath, low_memory=False)
    except Exception as e:
        print(f"  ✗ Read error: {e}")
        return None

    df.columns = df.columns.str.strip().str.lower()

    required = {"gas", "emissions_quantity", "start_time"}
    if not required.issubset(df.columns):
        print(f"  ✗ Missing columns: {required - set(df.columns)}")
        return None

    # Show what gas values actually exist (helps debug mismatches)
    unique_gases = df["gas"].dropna().unique()[:8]

    df = df[df["gas"].map(normalize_gas).isin(GAS_VARIANTS)]
    if df.empty:
        print(f"  ✗ No co2e_100yr rows. Gas values in file: {unique_gases}")
        return None

    df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce", utc=True)
    df = df[
        (df["start_time"] >= pd.Timestamp(YEAR_START, tz="UTC")) &
        (df["start_time"] <= pd.Timestamp(YEAR_END,   tz="UTC"))
    ]
    if df.empty:
        print(f"  ✗ No rows in 2020-2021 date range")
        return None

    df["emissions_quantity"] = pd.to_numeric(df["emissions_quantity"], errors="coerce")

    subsector = str(df["subsector"].iloc[0]) if "subsector" in df.columns else ""
    if subsector in KG_SUBSECTORS:
        df["emissions_quantity"] /= 1000.0

    df["year"] = df["start_time"].dt.year

    for col in ["iso3_country", "sector", "subsector",
                "temporal_granularity", "geometry_ref", "lat", "lon"]:
        if col not in df.columns:
            df[col] = np.nan

    df["geometry_ref"] = df["geometry_ref"].astype(str)

    keep = ["iso3_country", "sector", "subsector", "lat", "lon",
            "geometry_ref", "year", "start_time",
            "emissions_quantity", "temporal_granularity"]
    df = df[[c for c in keep if c in df.columns]].copy()
    df["source_file"] = Path(filepath).name
    return df


def enrich_with_geometry(df: pd.DataFrame, geo: gpd.GeoDataFrame | None) -> pd.DataFrame:
    if geo is None:
        df["lat"] = pd.to_numeric(df.get("lat", np.nan), errors="coerce")
        df["lon"] = pd.to_numeric(df.get("lon", np.nan), errors="coerce")
        return df

    if geo["geometry_ref"].notna().any() and "geometry_ref" in df.columns:
        geo_ref = geo[geo["geometry_ref"].notna()].copy()
        geo_ref = geo_ref.rename(columns={"_lat": "lat_geo", "_lon": "lon_geo"})
        geo_lookup = geo_ref[["geometry_ref", "lat_geo", "lon_geo"]].drop_duplicates("geometry_ref")

        df = df.merge(geo_lookup, on="geometry_ref", how="left")
        df["lat"] = df["lat_geo"].fillna(pd.to_numeric(df.get("lat", np.nan), errors="coerce"))
        df["lon"] = df["lon_geo"].fillna(pd.to_numeric(df.get("lon", np.nan), errors="coerce"))
        df = df.drop(columns=["lat_geo", "lon_geo"], errors="ignore")

        matched = df["lat"].notna().sum()
        print(f"  🔗 Ref join: {matched:,}/{len(df):,} matched")
        return df

    df["lat"] = pd.to_numeric(df.get("lat", np.nan), errors="coerce")
    df["lon"] = pd.to_numeric(df.get("lon", np.nan), errors="coerce")
    return df


def impute_missing_months(df: pd.DataFrame) -> pd.DataFrame:
    mask_monthly = (
        df["temporal_granularity"].astype(str).str.lower().str.contains("month", na=False)
    )
    monthly = df[mask_monthly].copy()
    other   = df[~mask_monthly].copy()

    if monthly.empty:
        return df

    monthly["month"] = monthly["start_time"].dt.month
    group_keys = ["iso3_country", "sector", "subsector", "lat", "lon", "year"]
    imputed_rows = []

    for keys, grp in monthly.groupby(group_keys, dropna=False):

        # ── FIX: collapse duplicate months by summing before reindexing ──
        month_series = (
            grp.groupby("month")["emissions_quantity"]
            .sum(min_count=1)          # NaN if all inputs NaN, else sum
        )
        # Now reindex safely — no duplicate month labels
        month_series = month_series.reindex(pd.RangeIndex(1, 13))

        if month_series.isna().all():
            continue

        # Impute missing months
        for m in pd.RangeIndex(1, 13)[month_series.isna()]:
            lower = month_series.iloc[:m-1].dropna()
            upper = month_series.iloc[m:].dropna()
            neighbours = []
            if not lower.empty: neighbours.append(lower.iloc[-1])
            if not upper.empty: neighbours.append(upper.iloc[0])

            if neighbours:
                month_series[m] = np.mean(neighbours)
            else:
                # Country-level fallback
                mask = (
                    (monthly["iso3_country"] == keys[0]) &
                    (monthly["sector"]        == keys[1]) &
                    (monthly["year"]          == keys[5])
                )
                month_series[m] = monthly.loc[mask, "emissions_quantity"].mean()

        # Reconstruct rows using metadata from first row of group
        meta = grp.iloc[0].drop(labels=["month", "emissions_quantity"], errors="ignore")
        for m, val in month_series.items():
            row = meta.copy()
            row["month"] = m
            row["emissions_quantity"] = val
            imputed_rows.append(row)

    monthly_imputed = pd.DataFrame(imputed_rows) if imputed_rows else monthly
    return pd.concat([other, monthly_imputed], ignore_index=True)


def aggregate_annual(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby(
            ["iso3_country", "sector", "subsector", "lat", "lon", "year"],
            dropna=False
        )["emissions_quantity"]
        .sum(min_count=1)
        .reset_index()
        .rename(columns={"emissions_quantity": "co2e_100yr_tonnes"})
    )

def impute_missing_months_fast(df: pd.DataFrame) -> pd.DataFrame:
    """
    Vectorised monthly imputation using pandas interpolate().
    ~100x faster than the loop version.
    """
    mask_monthly = (
        df["temporal_granularity"]
        .astype(str).str.lower()
        .str.contains("month", na=False)
    )
    monthly = df[mask_monthly].copy()
    other   = df[~mask_monthly].copy()

    if monthly.empty:
        return df

    monthly["month"] = monthly["start_time"].dt.month

    # Group key
    group_keys = ["iso3_country", "sector", "subsector", "lat", "lon", "year"]

    # Pivot to wide format: rows=groups, cols=months 1-12
    # This lets us interpolate across months vectorially
    monthly["grp"] = monthly.groupby(group_keys, dropna=False).ngroup()

    pivot = (
        monthly
        .groupby(["grp", "month"])["emissions_quantity"]
        .mean()                          # handles duplicate months
        .unstack(level="month")          # → wide: grp × month(1-12)
        .reindex(columns=range(1, 13))   # ensure all 12 months present
    )

    # ── Interpolate across months (linear, nearest neighbours) ───────────
    # limit_direction='both' fills edges too
    pivot_interp = pivot.interpolate(
        method="linear",
        axis=1,
        limit_direction="both"
    )

    # ── Fallback: groups still all-NaN → fill with country-level mean ────
    # Build country lookup from group metadata
    grp_meta = (
        monthly[group_keys + ["grp"]]
        .drop_duplicates("grp")
        .set_index("grp")
    )
    pivot_interp = pivot_interp.join(grp_meta[["iso3_country", "sector", "year"]])

    still_null = pivot_interp.iloc[:, :12].isna().any(axis=1)
    if still_null.any():
        country_means = (
            monthly.groupby(["iso3_country", "sector", "year"])["emissions_quantity"]
            .mean()
            .rename("country_mean")
        )
        pivot_interp = pivot_interp.join(
            country_means,
            on=["iso3_country", "sector", "year"]
        )
        for col in range(1, 13):
            mask = pivot_interp[col].isna()
            pivot_interp.loc[mask, col] = pivot_interp.loc[mask, "country_mean"]

        pivot_interp = pivot_interp.drop(
            columns=["iso3_country", "sector", "year", "country_mean"],
            errors="ignore"
        )
    else:
        pivot_interp = pivot_interp.drop(
            columns=["iso3_country", "sector", "year"],
            errors="ignore"
        )

    # ── Melt back to long format ──────────────────────────────────────────
    pivot_interp.index.name = "grp"
    long = (
        pivot_interp
        .reset_index()
        .melt(id_vars="grp", var_name="month", value_name="emissions_quantity")
    )

    # Merge metadata back
    long = long.merge(grp_meta.reset_index(), on="grp", how="left")

    # Reconstruct start_time from year + month
    long["start_time"] = pd.to_datetime(
        long["year"].astype(str) + "-" +
        long["month"].astype(str).str.zfill(2) + "-01",
        utc=True
    )
    long["temporal_granularity"] = "monthly"
    long = long.drop(columns=["grp"], errors="ignore")

    return pd.concat([other, long], ignore_index=True)
    

In [ ]:

# ── MAIN ──────────────────────────────────────────────────────────────────────

# Load geopackage once — shared across all CSVs
geo = None
gpkg_path = DATA_DIR / "geometries.gpkg"
if gpkg_path.exists():
    print(f"Loading shared geopackage...")
    geo = load_geopackage(str(gpkg_path))
else:
    print("⚠ No geometries.gpkg found — will use CSV lat/lon")

pairs = find_file_pairs(DATA_DIR)
frames = []

for pair in pairs:
    csv_path = pair["csv"]
    name = Path(csv_path).name
    print(f"\n📄 {name}")

    df = load_csv(csv_path)
    if df is None:
        continue

    df = enrich_with_geometry(df, geo)

    before = len(df)
    df = df.dropna(subset=["lat", "lon"])
    dropped = before - len(df)
    if dropped:
        print(f"  ⚠ Dropped {dropped:,} rows (no coordinates)")

    if df.empty:
        print(f"  ✗ Empty after coordinate filter")
        continue

    print(f"  ✓ {len(df):,} rows kept")
    frames.append(df)

print(f"\n{'─'*60}")
print(f"Files with usable data: {len(frames)}")

if not frames:
    print("✗ No usable data found.")
    print(f"\nDebug — first CSV columns:")
    test = pd.read_csv(list(DATA_DIR.glob("*.csv"))[0], nrows=3)
    print(test.columns.tolist())
    print(test.head(3))
else:
    raw = pd.concat(frames, ignore_index=True)
    print(f"Total rows:            {len(raw):,}")


In [16]:
print("\nImputing missing months...")
raw = impute_missing_months_fast(raw)
print(f"After imputation:      {len(raw):,}")



Imputing missing months...
After imputation:      4,627,260


In [17]:
print("\nAggregating to annual totals...")
final = aggregate_annual(raw)
print(f"Output rows:           {len(final):,}")



Aggregating to annual totals...
Output rows:           385,605


In [18]:

final.to_csv(OUTPUT_CSV, index=False)
print(f"\n✓ Saved → {OUTPUT_CSV}")
print(final.head(10).to_string(index=False))


✓ Saved → /lisc/home/user/gerardursin/pain-world/climate_trace_compiled.csv
iso3_country                sector                           subsector       lat        lon  year  co2e_100yr_tonnes
         ABW           agriculture                       crop-residues 12.530011 -69.977405  2021           0.000000
         ABW           agriculture                      cropland-fires 12.530011 -69.977405  2021           0.000000
         ABW           agriculture enteric-fermentation-cattle-pasture 12.530011 -69.977405  2021           0.000000
         ABW           agriculture             manure-applied-to-soils 12.530011 -69.977405  2021           0.000000
         ABW             buildings   non-residential-onsite-fuel-usage 12.530011 -69.977405  2021        5579.914000
         ABW             buildings       residential-onsite-fuel-usage 12.530011 -69.977405  2021       36659.249572
         ABW forestry-and-land-use                forest-land-clearing 12.530011 -69.977405  2021       